In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/venkat23f1000054/tfidf-baseline-model/tfidf_model.pkl
/kaggle/input/datasets/venkat23f1000054/tfidf-baseline-model/tfidf_vectorizer.pkl
/kaggle/input/datasets/venkat23f1000054/bilstm-model/bilstm_model.pt
/kaggle/input/datasets/venkat23f1000054/mlp-minilm-model/mlp_model.pt
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [4]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

Train Shape: (2000, 8)
Test Shape: (500, 7)


In [6]:
import torch
import torch.nn as nn

In [7]:
class BiLSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=128,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )
        self.fc = nn.Sequential(
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self,x):
        x = x.unsqueeze(-1)
        out,_ = self.lstm(x)
        out = out[:,-1,:]
        return self.fc(out)

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = BiLSTMClassifier().to(device)
model.load_state_dict(
    torch.load(
        "/kaggle/input/datasets/venkat23f1000054/bilstm-model/bilstm_model.pt",
        map_location=device
    )
)
model.eval()

BiLSTMClassifier(
  (lstm): LSTM(1, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [9]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
options = ["A","B","C","D","E"]
predictions = []
for _, row in test.iterrows():
    texts = [
        row["prompt"] + " [SEP] " + str(row[o])
        for o in options
    ]
    emb = embedder.encode(texts)
    X = torch.tensor(
        emb,
        dtype=torch.float32
    ).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(
            model(X)
        ).squeeze().cpu().numpy()
    ranked = np.argsort(probs)[::-1]
    top3 = [options[i] for i in ranked[:3]]
    predictions.append(
        " ".join(top3)
    )

In [11]:
sample = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)
sample["Prediction"] = predictions
sample.head()

,ID,Prediction
0,1,C A B
1,2,B D A
2,3,E D C
3,4,B D C
4,5,A C E


In [13]:
sample.to_csv("submission1.csv", index=False)